# IPCA e comunicação do Copom — coleta e análise exploratória

Notebook reprodutível para coletar as tabelas oficiais do IBGE/SIDRA e do Banco Central do Brasil e executar a EDA do painel mensal. O painel cobre as áreas geográficas N6 e N7 publicadas pelo SIDRA, os nove grupos principais de despesa e o período disponível desde janeiro de 2012. A data final é determinada pela atualização das tabelas.

Os códigos de ausência do SIDRA são mantidos como nulos; não há imputação. A regra IQR apenas sinaliza valores extremos, que permanecem na base. STL usa período sazonal 12; testes são executados no trecho observado contínuo de cada série, sem preencher lacunas internas. Expectativas Focus ficam em tabela separada com data de coleta e mês de referência para prevenir vazamento temporal.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f"Projeto: {ROOT}")

## 1. Coleta e preparação

A célula seguinte consulta as APIs oficiais e atualiza os CSVs em `dados/tratados/`. As expectativas Focus são snapshots mensais da última data disponível nos dez últimos dias de cada mês.

In [ ]:
from src.coleta_dados import coletar_base
from src.analise_eda import gerar_eda

ipca, macro, focus = coletar_base()
resultados = gerar_eda()

## 2. Cobertura e qualidade dos dados

In [ ]:
import pandas as pd
from IPython.display import display

cobertura = pd.DataFrame([
    {"base": "IPCA", "linhas": len(ipca), "inicio": ipca.data.min(), "fim": ipca.data.max(), "areas": ipca.area_id.nunique(), "grupos": ipca.grupo_id.nunique()},
    {"base": "Macroeconomia", "linhas": len(macro), "inicio": macro.data.min(), "fim": macro.data.max(), "areas": 1, "grupos": len(macro.columns) - 1},
    {"base": "Focus", "linhas": len(focus), "inicio": focus.data_coleta.min() if not focus.empty else pd.NaT, "fim": focus.data_coleta.max() if not focus.empty else pd.NaT, "areas": 1, "grupos": focus.mes_referencia.nunique() if not focus.empty else 0},
])
display(cobertura)
display(resultados["resumo_descritivo_unidade_grupo"].head(10))
display(resultados["perfil_ausentes"].sort_values("n_ausentes", ascending=False).head(15))

## 3. Estacionariedade, valores extremos e estrutura temporal

ADF tem hipótese nula de raiz unitária; KPSS tem hipótese nula de estacionariedade. Resultados devem ser interpretados em conjunto. As observações sinalizadas pelo IQR não foram removidas. ACF/PACF e componentes STL são calculados no trecho observado contínuo, sem imputar lacunas internas.

In [ ]:
testes = resultados["testes_estacionariedade"]
display(testes["status"].value_counts().rename_axis("status").to_frame("series"))
display(testes.loc[testes.status == "ok", ["area", "grupo_despesa", "adf_pvalor", "adf_conclusao_5pct", "kpss_pvalor", "kpss_conclusao_5pct"]].head(20))
display(resultados["outliers_iqr"].head(20))
display(resultados["resumo_decomposicao_stl"]["status"].value_counts().rename_axis("status").to_frame("series"))

## 4. Gráficos

As linhas finas mostram as áreas individuais; a linha destacada mostra a mediana entre áreas. A decomposição e ACF/PACF abaixo são exemplos de séries individuais; tabelas com diagnósticos por série estão salvas em `dados/tratados/`.

In [ ]:
from IPython.display import Image, display

for nome in ["ipca_por_grupo.png", "variaveis_macro.png", "decomposicao_stl_exemplo.png", "acf_pacf_exemplo.png"]:
    caminho = ROOT / "figuras" / nome
    if caminho.exists():
        print(nome)
        display(Image(filename=str(caminho)))

## Resultados desta extração

Extração realizada em 4 de novembro de 2026; o último mês disponível nas tabelas do SIDRA é setembro de 2026.

- O painel tem **25.488 linhas** (16 áreas × 9 grupos × 177 meses), de janeiro de 2012 a setembro de 2026. Cada série área × grupo contém entre **101 e 177 observações válidas**; os testes usam esse trecho contínuo sem imputação.
- Há **2.484 valores ausentes (9,75%)**, concentrados em períodos anteriores à disponibilidade de algumas áreas. O painel macroeconômico tem seis séries nacionais; o arquivo Focus guarda **6.906 expectativas** com data de coleta e referência separadas.
- A média observada é maior em **Alimentação e bebidas (0,57% ao mês)** e **Educação (0,53%)**, e menor em **Comunicação (0,11%)**. A dispersão é alta em Educação, Habitação e Transportes, com valores mensais no conjunto entre -8,62% e 9,37%.
- A regra IQR sinalizou **1.239 observações**, sobretudo em Educação (357), Comunicação (203) e Habitação (192). São alertas para contextualização, não exclusões.
- Nos 144 testes por série, ADF rejeitou raiz unitária em 104; KPSS rejeitou estacionariedade em 10, com 5 séries em que ambos rejeitam suas hipóteses nulas. A leitura é heterogênea: avaliar cada série e possíveis quebras estruturais antes de escolher transformações.
- A decomposição STL mostra maior desvio-padrão do componente sazonal em Educação, Habitação e Transportes. As tabelas detalhadas por área e grupo estão em `dados/tratados/`.

## Fontes e limitações

- IPCA: [SIDRA 1419](https://sidra.ibge.gov.br/tabela/1419) e [SIDRA 7060](https://sidra.ibge.gov.br/tabela/7060), variável 63.
- Explicativas: séries SGS 4189, 3697, 27574, 24364, 24369 e 433; expectativas do sistema Focus do BCB.
- A API Focus contém vintages e bases de cálculo distintas; não se deve juntar apenas pelo mês de referência sem respeitar a data de coleta.
- Este notebook cobre coleta, estruturação e EDA quantitativa. Coleta de notícias/documentos, classificação de sentimento e validação humana serão tratadas na etapa de texto, não simuladas neste conjunto.